# FRED Indicators Query — CPI, Inflation, and Core Macro Series

This notebook exercises `src/fred_api`, a standalone client for FRED's REST API
(`api.stlouisfed.org`), kept separate from the `macro_data` package (which fetches
only two FRED series today, `us_cpi` and `fed_funds_rate`, through the
catalog/schema/store pipeline — see `macro_data/sources/fred.py`). Nothing queried
here is written to `data/`; this is ad-hoc exploration, same spirit as
`BOT_query.ipynb` and `worldbank_gep_forecast.ipynb`.

Sections:
1. Setup — construct a `FREDClient`, define a `fetch_series()` helper
2. CPI — headline (`CPIAUCSL`) vs. core (`CPILFESL`), index levels
3. Inflation, three ways — headline CPI, core CPI, and PCE (`PCEPI`) year-over-year %
4. Unemployment rate (`UNRATE`)
5. Fed funds rate (`FEDFUNDS`) vs. headline inflation
6. Real GDP growth (`GDPC1`), year-over-year %
7. Notes — scope and how to get ongoing tracking instead of ad-hoc queries

## 1. Setup

`FREDClient()` reads `FRED_API_KEY` from the environment (same key `macro_data`'s
`fred` source uses). `fetch_series()` wraps `client.series.observations()` and cleans
the result: FRED returns `date`/`value` as strings (with `"."` for missing
observations in general, though none of the series below hit that), so this converts
types, drops unparseable rows, and sets `date` as the index — the notebook's own
light cleanup, not `macro_data.schema.normalize` (that function is internal to the
package pipeline and isn't imported here).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from fred_api import FREDClient

load_dotenv(Path("..") / ".env")
pd.set_option("display.max_columns", 10)
%matplotlib inline

fred = FREDClient()  # reads FRED_API_KEY from the environment
print(f"FREDClient ready, base_url={fred.base_url}")


def fetch_series(series_id: str, from_date: str = "2000-01-01") -> pd.DataFrame:
    """Fetch a FRED series and return it indexed by date with a numeric value column."""
    df = fred.series.observations(series_id, from_date=from_date)
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"]).set_index("date")
    return df[["value"]]

FREDClient ready, base_url=https://api.stlouisfed.org/fred/
